# CI/CD Pipelines {background-color="black" background-image="https://images.unsplash.com/photo-1556075798-4825dfaaf498?w=1600" background-size="cover" background-opacity="0.5"}


## The story so far

::::{.columns}

::: {.fragment .column width="50%"}

**Great tools give great responsibilities**

| Step | Tools | Result |
|------|-------| --------|
| IAC | OpenTofu, CloudFormation | VMs provisioned (multipass, AWS) |
| Configuration Management | Ansible | Services configured |

:::

::: {.fragment .column width="50%"}

**And a system to manage it**

Who *runs* `tofu apply` and `ansible-playbook` at scale?

- **You?**: doesn't scale
- **A cron job?**: no context, no approval, no state, no logs, no rollback -> NO
- **A pipeline**: triggered, reproducible, auditable ✅

:::
::::

# CI/CD Concepts {background-color="#1A1A2E"}

## What is CI/CD?
<https://www.redhat.com/en/topics/devops/what-is-ci-cd>

::::{.columns}

::: {.fragment .column width="35%"}

**Continuous Integration (CI)**

A commit can trigger an automated workflow:

1. Pull the code
2. Setup the environment (dependencies, secrets, …)
3. Run tests
4. Build the artefact (Docker image, binary, …)

:::{.fragment}
> CI can be thought of as a solution to the problem of having too many branches of an app in development at once that might conflict with each other.
:::
:::

::: {.fragment .column width="65%"}

**Continuous Delivery / Deployment (CD)**

Validated artefacts are automatically promoted:

- **Delivery** — ready to deploy at any time; a human approves the final push
- **Deployment** — fully automated, goes straight to production on merge

:::{.fragment}
> Continuous delivery usually means a developer’s changes to an application are automatically bug tested and uploaded to a repository (like GitHub or a container registry), where they can then be deployed to a live production environment by the operations team. 

> Continuous deployment is an extension of continuous delivery, and can refer to automating the release of a developer’s changes from the repository to production, where it is usable by customers.
:::
:::
::::

## CI/CD and IaC: the GitOps pattern

::::{.columns}

::: {.fragment .column width="55%"}

**GitOps principle:**  
The Git repository is the *single source of truth* for both application code **and** infrastructure.

```{mermaid}
flowchart TD
    PR["Pull Request: update main.tf"] --> CI
    CI["Pipeline: tofu plan output as PR comment"] --> MR["Merge to main"]
    MR --> CD["Pipeline: tofu apply"]
    CD --> INFRA["Infrastructure updated"]
```

:::

::: {.fragment .column width="45%"}

**Why it matters:**

- Every infra change has a PR, a review, and a commit
- Rollback = `git revert`
- Audit trail built-in
- No manual `tofu apply` from a developer's laptop

::: {.callout-tip}
This is how production teams manage Terraform at scale — the pipeline **is** the operator.
:::

:::
::::

# Gitea {background-color="#161b22" background-image="https://about.gitea.com/gitea.svg" background-size="15%" background-position="88% 50%"}

## Self-hosted CI/CD

::::{.columns}

::: {.fragment .column width="50%"}

**Scenarios where self-hosting makes sense:**

- **Air-gapped environments** — government, finance, military: no internet access allowed
- **Data sovereignty** — code and secrets never leave your infrastructure
- **Cost control** — no per-seat or per-minute charges
- **Learning & labs** — full CI/CD stack running on your laptop without cloud accounts
- **Edge / IoT** — pipelines that build firmware on-premises

:::

::: {.fragment .column width="50%"}

**Popular open-source options:**

| Tool | Notes |
|------|-------|
| **Gitea** + Gitea Actions | Lightweight, GitHub-compatible, ~300 MB RAM |
| **Forgejo** | Gitea fork, community-governed, same syntax |
| GitLab CE | Full-featured but heavy (4 GB+ RAM) |
| Jenkins | Classic, very extensible, XML-based config |
| Drone CI | Docker-native, pipeline-as-code |

> We will use **Gitea Actions** — same YAML as GitHub Actions, runs in Docker Compose.

:::
::::

## What is Gitea?

::::{.columns}

::: {.fragment .column width="55%"}

**Gitea** is a self-hosted Git service — a lightweight GitHub/GitLab alternative:

- Repositories, issues, pull requests, wikis, packages
- **Gitea Actions:** CI/CD engine with GitHub Actions-compatible YAML
- Written in Go — single binary, low resource footprint
- Deploy with a single Docker Compose file

::: {.callout-tip}
**Key insight:** Gitea Actions workflows use the **exact same syntax** as GitHub Actions.  
A workflow that runs in Gitea can be moved to GitHub with minimal changes.
:::

:::

::: {.fragment .column width="45%"}

```{mermaid}
flowchart TD
    DEV["git push"] --> GITEA["Gitea Server\n(port 3000)"]
    GITEA -->|"trigger"| RUNNER["Gitea Runner\n(act_runner)"]
    RUNNER -->|"spin up"| DOCKER["Docker container\n(ubuntu-latest image)"]
    DOCKER --> STEPS["Run workflow steps"]
```

:::
::::

## Install Gitea on the host

<https://docs.gitea.com/category/installation> 

::::{.columns}

::: {.fragment .column width="50%"}

**Download and install the Gitea binary:**

```bash
cd lab/gitea-setup
./install-gitea.sh
```

:::

::: {.fragment .column width="50%"}

**First-run setup wizard:**

```
http://localhost:3000
→ Database: SQLite3
→ Site title: My Gitea
→ Admin user: admin / <password>
→ Click "Install Gitea"
```

::: {.callout-tip}
Same approach as our other tools:
`tofu` via snap, `ansible` via apt, now `gitea` as a binary — no Docker required.
:::

::: {.callout-note}
Enable Actions in the admin panel:  
**Site Administration → Configuration → Actions → Enable**
:::

:::
::::


## Install and register the runner

::::{.columns}

::: {.fragment .column width="50%"}


**1. Generate a registration token in Gitea UI:**

```
Site Administration
  → Actions → Runners
    → Create new runner → Copy token
```

**2. Create environment variables:**

```bash
export GITEA_TOKEN="<token-from-gitea-ui>"
``` 

**3. Register and start:**

```bash
cd lab/gitea-setup
./register-runner.sh  
```
:::

::: {.fragment .column width="50%"}

**4. Verify the runner is connected:**

```
Gitea UI → Site Admin → Actions → Runners
# Should show: host-runner  Online  ✅
```

:::
::::


## Gitea Actions — Provision Multipass VMs with OpenTofu

**`.gitea/workflows/provision.yml`**

```yaml
name: Provision Multipass VMs

on:
  push:
    branches: [main]
    paths:
      - 'terraform/**'
      - 'ansible/**'
  workflow_dispatch:

jobs:
  provision:
    runs-on: self-hosted        # host runner — multipass lives here
    steps:
      - uses: actions/checkout@v4

      - name: Write SSH key pair
        run: |
          echo "${{ secrets.SSH_PRIVATE_KEY }}" > terraform/id_ed25519
          echo "${{ secrets.SSH_PUBLIC_KEY }}"  > terraform/id_ed25519.pub
          chmod 600 terraform/id_ed25519

      - name: tofu init
        run: tofu -chdir=terraform init

      - name: tofu plan
        run: tofu -chdir=terraform plan -out=tfplan

      - name: tofu apply
        run: tofu -chdir=terraform apply -auto-approve tfplan

  configure:
    runs-on: self-hosted
    needs: provision            # wait for VMs to be ready
    steps:
      - uses: actions/checkout@v4

      - name: Restore SSH key
        run: |
          echo "${{ secrets.SSH_PRIVATE_KEY }}" > terraform/id_ed25519
          chmod 600 terraform/id_ed25519

      - name: Wait for SSH
        run: sleep 20

      - name: Install Docker via Ansible
        run: |
          ansible-playbook \
            -i ansible/hosts.ini \
            ansible/install_docker.yml
```


### Running the pipeline

::::{.columns}

::: {.fragment .column width="50%"}

**1. Generate SSH key pair (once):**

```bash
# in lab/gitea-multipass-terraform/
ssh-keygen -t ed25519 -f terraform/id_ed25519 -N ""
```

**2. Create the repo and set secrets:**

```bash
# in lab/gitea-setup/
GITEA_TOKEN=<admin-token> ./setup-pipeline.sh
```

**3. Push the code:**

```bash
# in lab/gitea-multipass-terraform/
git init
git remote add gitea http://localhost:3000/admin/infra.git
git add .
git commit -m "initial commit"
git push gitea main
```

:::

::: {.fragment .column width="50%"}

**4. Watch the pipeline run:**

```
http://localhost:3000/admin/infra
  → Actions tab

Run #1  Provision Multipass VMs
  ├─ provision  ✅
  │    ├─ tofu init
  │    ├─ tofu plan
  │    └─ tofu apply   (manager + worker1 + worker2)
  └─ configure  ✅
       ├─ Wait for SSH
       └─ Install Docker via Ansible
```

**5. Verify Docker is running on the VMs:**

```bash
multipass exec manager -- docker info | grep 'Server Version'
multipass exec worker1 -- docker ps
```

**Key points:**
- Two chained jobs: `provision` → `configure` via `needs:`
- `tofu apply` writes `ansible/hosts.ini` automatically
- SSH keys live in **Gitea Secrets**, never in the repo

:::
::::


### GitOps in action — change and re-deploy

::::{.columns}

::: {.fragment .column width="50%"}

**Change something and push to trigger the pipeline:**

```bash
# Add a worker — edit terraform/variables.tf
# Change: worker_count = 2  →  worker_count = 3

cd lab/gitea-multipass-terraform
git add terraform/variables.tf
git commit -m "scale: add worker3"
git push gitea main
```

The pipeline triggers automatically on push.

:::

::: {.fragment .column width="50%"}

**Observe the GitOps flow:**

```
git push
  └─► Gitea detects change in terraform/**
        └─► Pipeline Run #2 starts
              ├─ provision ✅
              │    └─ tofu apply  (adds worker3)
              └─ configure ✅
                   └─ Docker installed on worker3
```

**Verify the new VM:**

```bash
multipass list
# NAME      STATE    IPV4
# manager   Running  10.x.x.x
# worker1   Running  10.x.x.x
# worker2   Running  10.x.x.x
# worker3   Running  10.x.x.x  ← new

multipass exec worker3 -- docker info | grep 'Server Version'
```

:::
::::


### Cleanup

::::{.columns}

::: {.fragment .column width="50%"}

**Destroy the VMs and reset the code repo:**

```bash
# Destroy VMs and remove generated files
cd lab/gitea-multipass-terraform
./cleanup.sh
```

This runs `tofu destroy`, then removes:
- Multipass instances (`manager`, `worker1`, `worker2`)
- SSH keys, `cloud-init.rendered.yml`, `hosts.ini`
- OpenTofu state and cache
- `.git/` — the canonical repo lives on Gitea

:::

::: {.fragment .column width="50%"}

**Reset Gitea to start over:**

```bash
# Stop Gitea + runner, wipe all data
cd lab/gitea-setup
./cleanup.sh
```

This stops the `gitea` and `act_runner` processes, then removes:
- `conf/`  `data/`  `log/`
- `.runner`  `gitea`  `act_runner` binaries

Run `./install-gitea.sh` to start fresh.

::: {.callout-warning}
`cleanup.sh` asks for confirmation before deleting data.
:::

:::
::::
